In [1]:
import os
import time
import pandas as pd
import requests

In [2]:
# path config

FINAL_DF_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_FINAL_DEDUPED.csv"
EXISTING_RESULTS_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_pilot_results.csv"
FULL_OUTPUT_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results.csv"

# Optional checkpoint file that updates during the run
CHECKPOINT_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results_checkpoint.csv"

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
if not API_KEY:
    raise ValueError("GOOGLE_MAPS_API_KEY not found in environment variables.")

SLEEP_SECONDS = 0.2
SAVE_EVERY = 100

In [ ]:
# load deduped phamacy list and existing queries
final_df = pd.read_csv(FINAL_DF_PATH)
existing_results = pd.read_csv(EXISTING_RESULTS_PATH)

print(f"Rows in final_df: {len(final_df):,}")
print(f"Rows in existing pilot results: {len(existing_results):,}")

Rows in final_df: 4,862
Rows in existing pilot results: 100


In [4]:
# helper functions
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None


def build_query(row):
    """
    Build the same query format used in the pilot.
    Keep this unchanged if you want query_string matching to work.
    """
    practice_name = clean_text(row.get("PRACTICE_NAME"))
    address = clean_text(row.get("ADDRESS"))
    city = clean_text(row.get("CITY"))
    province = clean_text(row.get("PROVINCE"))

    parts = []

    if practice_name:
        parts.append(practice_name)
    if address:
        parts.append(address)
    if city:
        parts.append(city)
    if province:
        parts.append(province)

    parts.append("South Africa")

    return ", ".join(parts)


def search_place_text(query, api_key):
    """
    Google Places API (Legacy) Text Search
    """
    url = "https://maps.googleapis.com/maps/api/place/textsearch/json"

    params = {
        "query": query,
        "type": "pharmacy",
        "key": api_key
    }

    r = requests.get(url, params=params, timeout=30)
    http_status = r.status_code

    try:
        data = r.json()
    except Exception:
        return {
            "http_status": http_status,
            "api_status": "BAD_JSON",
            "place_id": None,
            "matched_name": None,
            "matched_address": None,
            "lat": None,
            "lng": None,
            "types": None,
            "raw_error": r.text[:500]
        }

    status = data.get("status")

    if status != "OK":
        return {
            "http_status": http_status,
            "api_status": status,
            "place_id": None,
            "matched_name": None,
            "matched_address": None,
            "lat": None,
            "lng": None,
            "types": None,
            "raw_error": str(data)[:500]
        }

    result = data["results"][0]

    return {
        "http_status": http_status,
        "api_status": status,
        "place_id": result.get("place_id"),
        "matched_name": result.get("name"),
        "matched_address": result.get("formatted_address"),
        "lat": result.get("geometry", {}).get("location", {}).get("lat"),
        "lng": result.get("geometry", {}).get("location", {}).get("lng"),
        "types": ", ".join(result.get("types", [])) if result.get("types") else None,
        "raw_error": None
    }


def needs_review(row):
    if row["api_status"] != "OK":
        return True
    if pd.isna(row["place_id"]):
        return True
    if pd.isna(row["lat"]) or pd.isna(row["lng"]):
        return True
    return False

In [5]:
# rebuild query string in final df
places_df = final_df.copy()
places_df["query_string"] = places_df.apply(build_query, axis=1)

# Remove blank query strings just in case
places_df = places_df[
    places_df["query_string"].notna() &
    (places_df["query_string"].str.len() > 0)
].copy()

print(f"Rows with usable query_string: {len(places_df):,}")


Rows with usable query_string: 4,862


In [6]:
# identify previous queries
if "query_string" not in existing_results.columns:
    raise ValueError("existing_results file does not contain a query_string column.")

already_done = set(existing_results["query_string"].dropna())

print(f"Unique query_strings already completed: {len(already_done):,}")

remaining_df = places_df[~places_df["query_string"].isin(already_done)].copy()

print(f"Rows remaining to query: {len(remaining_df):,}")

Unique query_strings already completed: 100
Rows remaining to query: 4,761


In [7]:
# query remaining records
results = []

for n, (_, row) in enumerate(remaining_df.iterrows(), start=1):
    query = row["query_string"]
    result = search_place_text(query, API_KEY)

    results.append({
        "PHARMACY_ID": row.get("PHARMACY_ID"),
        "PRACTICE_NUM": row.get("PRACTICE_NUM"),
        "PRACTICE_NAME": row.get("PRACTICE_NAME"),
        "ADDRESS": row.get("ADDRESS"),
        "CITY": row.get("CITY"),
        "PROVINCE": row.get("PROVINCE"),
        "PHONE": row.get("PHONE"),
        "query_string": query,
        **result
    })

    if n % 100 == 0:
        print(f"Processed {n:,} / {len(remaining_df):,}")

    # checkpoint save every SAVE_EVERY rows
    if n % SAVE_EVERY == 0:
        checkpoint_new = pd.DataFrame(results)

        if len(checkpoint_new) > 0:
            checkpoint_new["needs_review"] = checkpoint_new.apply(needs_review, axis=1)

        checkpoint_combined = pd.concat([existing_results, checkpoint_new], ignore_index=True)
        checkpoint_combined = checkpoint_combined.drop_duplicates(subset=["query_string"], keep="first")
        checkpoint_combined.to_csv(CHECKPOINT_PATH, index=False)

        print(f"Checkpoint saved: {CHECKPOINT_PATH}")

    time.sleep(SLEEP_SECONDS)


Processed 100 / 4,761
Checkpoint saved: C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results_checkpoint.csv
Processed 200 / 4,761
Checkpoint saved: C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results_checkpoint.csv
Processed 300 / 4,761
Checkpoint saved: C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results_checkpoint.csv
Processed 400 / 4,761
Checkpoint saved: C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results_checkpoint.csv
Processed 500 / 4,761
Checkpoint saved: C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results_checkpoint.csv
Processed 600 / 4,761
Checkpoint saved: C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results_checkpoint.csv
Processed 700 / 4,761
Checkpoint saved: C:\Users\jcahi\Box\DAIR_SA\Data Sets

In [8]:
# new reults df
new_results = pd.DataFrame(results)

if len(new_results) > 0:
    new_results["needs_review"] = new_results.apply(needs_review, axis=1)
else:
    new_results["needs_review"] = pd.Series(dtype=bool)

print(f"New rows queried this run: {len(new_results):,}")

New rows queried this run: 4,761


In [9]:
# combine results
combined_results = pd.concat([existing_results, new_results], ignore_index=True)
combined_results = combined_results.drop_duplicates(subset=["query_string"], keep="first")

print(f"Combined total rows: {len(combined_results):,}")

Combined total rows: 4,812


In [10]:
# save final results
combined_results.to_csv(FULL_OUTPUT_PATH, index=False)

print("\nDone.")
print(f"Saved full combined results to:\n{FULL_OUTPUT_PATH}")

print("\nAPI status counts:")
print(combined_results["api_status"].value_counts(dropna=False))

print("\nNeeds review counts:")
print(combined_results["needs_review"].value_counts(dropna=False))


Done.
Saved full combined results to:
C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_places_full_results.csv

API status counts:
OK                   4539
ZERO_RESULTS          272
DEADLINE_EXCEEDED       1
Name: api_status, dtype: int64

Needs review counts:
False    4539
True      273
Name: needs_review, dtype: int64
